In [38]:
import sqlite3
import pandas as pd
con=sqlite3.connect('../Data/raw/airtel.db')

def q(sql):
    return pd.read_sql(sql, con)

In [39]:
def run_sql_file(path):
    with open(path, 'r') as f:
        script = f.read()
    con.executescript(script)
    con.commit()
    print('ran',path)

In [40]:
# A1 · How many subscribers are prepaid and 
# how many postpaid, and what share is each?
# STEPS
# 1. Open customer_features, where one row is one subscriber.
# 2. Group by connection_type.
# 3. Count the rows in each group.
# 4. Express each count as a share of 22,000. 

q("""
SELECT
    connection_type,
    COUNT(*) AS subscriber_count,
    ROUND((COUNT(*) * 100.0 / 22000), 2) AS perc
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subscriber_count,perc
0,Postpaid,4851,22.05
1,Prepaid,17149,77.95


In [41]:
"""A2 · Which five plans have the most subscribers, and what does each cost?
STEPS
1. Group customer_features by plan_name.
2. Carry plan_type and price_inr through the group.
3. Count and sort, biggest first.
4. Show the top five."""

q("""
SELECT
    plan_name,
    plan_type,
    price_inr,
    COUNT(*) AS subs
FROM customer_features
GROUP BY plan_name, plan_type, price_inr
ORDER BY subs DESC
LIMIT 5;
""")

,plan_name,plan_type,price_inr,subs
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


In [42]:
"""A3 · Which five circles have the most subscribers?
STEPS
1. Group by circle_name, carrying region.
2. Count.
3. Sort and take five."""

q("""
SELECT
    circle_name,
    region,
    COUNT(*) AS subscriber_count
FROM customer_features
GROUP BY circle_name, region
ORDER BY subscriber_count DESC
LIMIT 5;
""")

,circle_name,region,subscriber_count
0,Maharashtra,West,1428
1,Andhra Pradesh,South,1412
2,Mumbai,West,1395
3,Karnataka,South,1361
4,North East,East,1318


In [43]:
"""A4 · How many subscribers are in each age band?
STEPS
1. Group by age_band.
2. Count and express as a share.
3. Sort by band, not by size, so the shape of the distribution is visible."""

q("""
SELECT
    age_band,
    COUNT(*) AS subs,
    ROUND((COUNT(*) * 100.0 / 22000), 2) AS perc
FROM customer_features
GROUP BY age_band
ORDER BY subs;
""")

,age_band,subs,perc
0,55+,1546,7.03
1,45-54,2895,13.16
2,35-44,4824,21.93
3,18-24,5268,23.95
4,25-34,7467,33.94


In [44]:
"""A5 · How many subscribers did each acquisition channel bring in?
STEPS
1. Group by acquisition_channel.
2. Count and share.
3. Sort biggest first."""

q("""
SELECT
    acquisition_channel,
    COUNT(*) AS subs,
    ROUND((COUNT(*) * 100.0 / 22000), 2) AS perc
from customer_features
GROUP BY acquisition_channel
ORDER BY subs DESC;
""")

,acquisition_channel,subs,perc
0,Retailer,9287,42.21
1,Airtel App,6513,29.60
2,Website,3081,14.00
3,Call Center,1770,8.05
4,Doorstep,1349,6.13


In [45]:
# A6 · How many subscribers use each device type and operating system?
# STEPS
# 1. Group by device_type and os together.
# 2. Count.
# 3. Read the result and notice that only three of the
# six possible combinations exist.

q("""
SELECT
    device_type,os,COUNT(*) AS subs
FROM customer_features
GROUP BY device_type, os
ORDER BY subs DESC;
""")

,device_type,os,subs
0,Smartphone,Android,15613
1,Smartphone,iOS,3358
2,Feature Phone,Other,3029


In [46]:
#A7 · How many subscribers have 0, 1, 2 and 3 completed years with us?
#S T E P S
#1. Group by tenure_years, which Step 2 calculated from activation_date.
#2. Count and share.
#3. Sort by year.

q("""
SELECT
    tenure_years, COUNT(*) AS subs,
    round((COUNT(*) * 100.0 / 22000), 2) AS perc
FROM customer_features
GROUP BY tenure_years
ORDER BY subs;
""")

,tenure_years,subs,perc
0,3,3374,15.34
1,0,4610,20.95
2,1,6961,31.64
3,2,7055,32.07


In [47]:
"""B1 · What is our average monthly revenue per subscriber,
 by the company's ARPU figure? — 

STEPS
1. Open customer_features.
2. Average stated_arpu.
3. Report it."""

q("""
SELECT ROUND(AVG(stated_arpu),2) AS avg_arpu
FROM customer_features
""")

,avg_arpu
0,421.65


In [48]:
# B2 · What is our average monthly revenue per subscriber, from the payments received?
# STEPS
# 1. Open customer_features again.
# 2. Average actual_arpu_6m, which Step 2 built from recharges.
# 3. Also total it, to get the monthly revenue of the whole base.

q("""
SELECT ROUND(AVG(actual_arpu_6m),2) AS avg_arpu,
       ROUND(SUM(actual_arpu_6m),2) AS total_revenue
FROM customer_features
""")

,avg_arpu,total_revenue
0,344.46,7578210.14


In [49]:
# B3 · By how much does the company's ARPU figure overstate the money received?
# STEPS
# 1. Average both columns in one query.
# 2. Average the gap.
# 3. Express the difference as a percentage of the actual total, not of the stated one.

q("""
SELECT ROUND(AVG(stated_arpu),2) AS avg_stated_arpu,
       ROUND(AVG(actual_arpu_6m),2) AS avg_actual_arpu,
       ROUND(AVG(arpu_gap),2) AS avg_gap,
       ROUND(100.0 * (SUM(stated_arpu) - 
       SUM(actual_arpu_6m)) / SUM(actual_arpu_6m), 2) AS gap_perc
FROM customer_features
""")

,avg_stated_arpu,avg_actual_arpu,avg_gap,gap_perc
0,421.65,344.46,77.18,22.41


In [50]:
# B4 · Is that gap the same for prepaid and for postpaid subscribers?
# STEPS
# 1. Group by connection_type.
# 2. Average both columns inside each group.
# 3. Read the two rows against each other rather than down the column.

q("""
SELECT connection_type,
       COUNT(*) AS subs,
       ROUND(AVG(stated_arpu),2) AS avg_stated_arpu,
       ROUND(AVG(actual_arpu_6m),2) AS avg_actual_arpu
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,subs,avg_stated_arpu,avg_actual_arpu
0,Postpaid,4851,959.65,970.42
1,Prepaid,17149,269.46,167.40


In [51]:
# B5 · Which five plans bring in the most actual monthly revenue?
# STEPS
# 1. Group by plan_name.
# 2. Sum actual_arpu_6m for total monthly revenue and average it for revenue per subscriber.
# 3. Sort by total, take five.

q("""
SELECT plan_name,COUNT(*) AS subs,
ROUND(SUM(actual_arpu_6m),2) AS total_rvenue,
ROUND(AVG(actual_arpu_6m),2) AS avg_revenue
FROM customer_features
GROUP BY plan_name
ORDER BY total_rvenue DESC
LIMIT 5;
""")

,plan_name,subs,total_rvenue,avg_revenue
0,Postpaid Infinity 1499,1431,2171756.20,1517.65
1,Postpaid Family 999,1239,1249245.94,1008.27
2,Postpaid 749,946,718958.94,760.00
3,Prepaid Entertainment 399,2142,671849.50,313.66
4,Prepaid Data 349,1884,520066.00,276.04


In [52]:
# 6 B6 · Which five circles bring in the most actual monthly revenue?
# STEPS
# 1. Group by circle_name.
# 2. Sum actual_arpu_6m.
# 3. Sort and take five.

q("""
SELECT circle_name,COUNT(*) AS subs,
ROUND(SUM(actual_arpu_6m),2) AS total_rvenue,
ROUND(AVG(actual_arpu_6m),2) AS avg_revenue
FROM customer_features
GROUP BY circle_name
ORDER BY total_rvenue DESC
LIMIT 5;
""")

,circle_name,subs,total_rvenue,avg_revenue
0,Mumbai,1395,473288.16,339.27
1,Karnataka,1361,468514.95,344.24
2,Andhra Pradesh,1412,463369.80,328.17
3,North East,1318,460998.29,349.77
4,Maharashtra,1428,450032.30,315.15


In [53]:
# B7 · How much money did we receive from January to June 2025?
# STEPS
# 1. This is a question about payments, not subscribers, so read recharges.
# 2. Filter to the window.
# 3. Count and sum.

q("""
SELECT COUNT(*) AS num_recharges,
SUM(amount_inr) AS total_revenue
FROM recharges
WHERE recharge_date BETWEEN '2025-01-01' AND '2025-06-30'
""")

,num_recharges,total_revenue
0,83693,45469268


In [54]:
# B8 · Do subscribers who have been with us longer pay more each month?
# STEPS
# 1. Group by tenure_years.
# 2. Average actual_arpu_6m.
# 3. Sort by year

q("""
SELECT tenure_years,COUNT(*) AS subs,
ROUND(AVG(actual_arpu_6m),2) AS avg_revenue
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years
""")

,tenure_years,subs,avg_revenue
0,0,4610,329.10
1,1,6961,342.72
2,2,7055,350.13
3,3,3374,357.22


In [55]:
# C1 · How many payments of each type did we take, and from how many subscribers?
# S T E P S
# 1. Read recharges, where one row is one payment.
# 2. Group by txn_type.
# 3. Count rows and distinct subscribers in each group.

q("""
SELECT txn_type,
    COUNT(*) AS num_payments,
    COUNT(DISTINCT customer_id) AS num_subscribers
FROM recharges
GROUP BY txn_type
""")

,txn_type,num_payments,num_subscribers
0,Bill Payment,108419,4851
1,Recharge,299670,17149


In [56]:
# C2 · What is the average value of a payment? — deliberate failure 2

# S T E P S
# 1. Read recharges.
# 2. Average amount_inr.

q("""
SELECT ROUND(AVG(amount_inr),2) AS avg_payment
FROM recharges
""")

,avg_payment
0,532.59


In [57]:
# C3 · What is the average payment for a prepaid recharge, and for a postpaid bill?
# S T E P S
# 1. Same table, same average.
# 2. Add GROUP BY txn_type.
# 3. Add the minimum and maximum so the two ranges can be compared.

q("""
SELECT txn_type, ROUND(AVG(amount_inr),2) avg,
         MIN(amount_inr) min,
         MAX(amount_inr) max
FROM recharges
GROUP BY txn_type
""")

,txn_type,avg,min,max
0,Bill Payment,1082.42,399,1873
1,Recharge,333.66,99,1799


In [58]:
# C4 · How many times did a prepaid and a postpaid subscriber pay in six months?
# S T E P S
# 1. Back to customer_features, because this is now a question about subscribers.
# 2. Group by connection_type.
# 3. Average recharge_count and recharge_total_6m.

q("""
SELECT connection_type,
    ROUND(AVG(recharge_count),2) AS avg_recharge_count,
    ROUND(AVG(recharge_total_6m),2) AS avg_recharge_total_6m
FROM customer_features
GROUP BY connection_type
""")

,connection_type,avg_recharge_count,avg_recharge_total_6m
0,Postpaid,5.39,5822.51
1,Prepaid,3.36,1004.39


In [59]:
# C5 · Which payment channels take the most payments?
# S T E P S
# 1. Read recharges.
# 2. Group by channel.
# 3. Count and share.

q("""
SELECT channel, COUNT(*) AS num_payments,
    ROUND(COUNT(*) * 100.0 / 408090,2) AS perc
FROM recharges
GROUP BY channel
ORDER BY num_payments DESC

""")

,channel,num_payments,perc
0,Website,111163,27.24
1,Airtel App,111083,27.22
2,USSD,75257,18.44
3,Retailer,74406,18.23
4,AutoPay,36180,8.87


In [60]:
# C6 · Do prepaid and postpaid subscribers pay through the same channels?
# STEPS
# 1. Group by txn_type and channel.
# 2. Use a window function to express each count as a share of its own txn_type, so the two groups can be 
# compared fairly.

q("""
SELECT txn_type, channel, COUNT(*) AS num_payments,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY txn_type), 2) AS percentage
FROM recharges
GROUP BY txn_type, channel
""")

,txn_type,channel,num_payments,percentage
0,Bill Payment,Airtel App,36186,33.38
1,Bill Payment,AutoPay,36180,33.37
2,Bill Payment,Website,36053,33.25
3,Recharge,Airtel App,74897,24.99
4,Recharge,Retailer,74406,24.83
5,Recharge,USSD,75257,25.11
6,Recharge,Website,75110,25.06


In [61]:
# C7 · Does the number of payments a subscriber made go with whether they left?
# STEPS
# 1. Group customer_features by recharge_count.
# 2. Count subscribers and average churn in each group.
# 3. Restrict to counts of eight or fewer, where the groups are large enough to read.

q("""
SELECT recharge_count, COUNT(*) AS subs,
    ROUND(AVG(is_churned),2) AS avg_churn
FROM customer_features
GROUP BY recharge_count
ORDER BY recharge_count;
""")

,recharge_count,subs,avg_churn
0,0,3711,0.44
1,1,2762,0.40
2,2,2339,0.28
3,3,1026,0.54
4,4,714,0.76
5,5,3407,0.14
6,6,6732,0.02
7,7,507,0.06
8,8,32,0.91
9,9,30,0.87


In [62]:
# D1 · How much data did the network carry each month, and for how many subscribers?
# STEPS
# 1. Read v_usage_month, where one row is one subscriber in one month.
# 2. Group by month.
# 3. Sum the data and convert megabytes to terabytes.
# 4. Also count the distinct subscribers who appear, because that number is going to matter.

q("""
SELECT month,
ROUND(SUM(total_mb) / 1024 / 1024, 2) AS data_tb,
COUNT(DISTINCT customer_id) AS subs
FROM v_usage_month
GROUP BY month
""")

,month,data_tb,subs
0,2025-01,337.97,21302
1,2025-02,307.32,21162
2,2025-03,332.51,20366
3,2025-04,310.27,19320
4,2025-05,363.10,18356
5,2025-06,338.95,17326


In [63]:
#  D2 · What is a subscriber's average daily data use?
# STEPS
# 1. Read usage_daily.
# 2. Average data_mb.

q("""
SELECT ROUND(AVG(data_mb),2) AS avg_daily_data
FROM usage_daily
""")

,avg_daily_data
0,1572.87


In [64]:
# D3 · What is the average data per subscriber, per active day and per observed day?
# STEPS
# 1. Read customer_features, where Step 2 already computed both.
# 2. Average each.
# 3. Divide one by the other.
q("""SELECT ROUND(AVG(mb_per_observed_day), 2) AS avg_obsrv_day,
    ROUND(AVG(mb_per_active_day), 2) AS avg_active_day
    FROM customer_features;""")

,avg_obsrv_day,avg_active_day
0,524.05,1404.44


In [65]:
# D4 · Does data use per day change with the day of the week?
# STEPS
# 1. Read usage_daily.
# 2. Use strftime('%w', usage_date) to pull the weekday out.
# 3. Group by it.
# 4. Count rows and average the data.

q("""
SELECT strftime('%w', usage_date) AS weekday,
    COUNT(*) AS num_rows,
    ROUND(AVG(data_mb), 2) AS avg_data_mb
FROM usage_daily
GROUP BY weekday;
""")

,weekday,num_rows,avg_data_mb
0,0,190368,1578.06
1,1,189914,1572.94
2,2,183823,1569.55
3,3,191088,1564.48
4,4,190764,1580.00
5,5,190542,1569.51
6,6,190240,1575.45


In [66]:
# D5 · How many subscribers used the network in each month?
# STEPS
# 1. Read v_usage_month.
# 2. Count distinct subscribers per month.
# 3. Read the trend rather than any single number.

q("""
SELECT month,
COUNT(DISTINCT customer_id) AS subs
FROM v_usage_month
GROUP BY month
ORDER BY month;
""")

,month,subs
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


In [67]:
#D6 · Did voice minutes and SMS per day change from January to June? 
# STEPS 
# 1. Read usage_daily. 
# 2. Bucket by month with strftime. 
# 3. Average voice_minutes and sms_count.
q("""
SELECT strftime('%Y-%m', usage_date) AS month,
    ROUND(AVG(voice_minutes), 2) AS avg_voice_minutes,
    ROUND(AVG(sms_count), 2) AS avg_sms_count
FROM usage_daily
GROUP BY month
""")

,month,avg_voice_minutes,avg_sms_count
0,2025-01,55.56,0.75
1,2025-02,55.84,0.75
2,2025-03,56.08,0.75
3,2025-04,56.49,0.76
4,2025-05,56.70,0.76
5,2025-06,57.13,0.77


In [68]:
#D7 · How much data did an active subscriber use each month, and on how many days?
#STEPS
#1. Read v_usage_month.
#2. Group by month.
#3. Average total_mb per subscriber and convert to gigabytes.
#4. Also average usage_days, to see whether people are appearing more often or using more when they 
#appear.

q("""
SELECT month,
    ROUND(AVG(total_mb) / 1024, 2) AS avg_data_gb,
    ROUND(AVG(usage_days), 2) AS avg_usage_days
FROM v_usage_month
GROUP BY month
""")

,month,avg_data_gb,avg_usage_days
0,2025-01,16.25,11.39
1,2025-02,14.87,10.31
2,2025-03,16.72,11.52
3,2025-04,16.44,11.25
4,2025-05,20.26,11.73
5,2025-06,20.03,11.46


In [69]:
# D8 · Among subscribers who never left, did data per active day still rise in May?
# Hold the cohort constant by keeping only subscribers who never churned.
q("""
SELECT
    v.month,
    COUNT(DISTINCT v.customer_id) AS subscribers,
    ROUND(AVG(1.0 * v.total_mb / NULLIF(v.usage_days, 0)), 2) AS avg_mb_per_active_day
FROM v_usage_month AS v
JOIN customer_features AS c
    ON c.customer_id = v.customer_id
WHERE c.is_churned = 0
GROUP BY v.month
ORDER BY v.month
""")

,month,subscribers,avg_mb_per_active_day
0,2025-01,16293,1410.58
1,2025-02,16588,1422.15
2,2025-03,16640,1418.06
3,2025-04,16648,1411.76
4,2025-05,16667,1659.45
5,2025-06,16660,1662.24


In [70]:
# E1 · Split into four equal groups by daily data, what range does each group cover?
# STEPS
# 1. Take customer_features, excluding the four subscribers with no usage at all.
# 2. Apply NTILE(4) ordered by mb_per_active_day.
# 3. Group by the quartile and report its size and range.


q("""
with n as (SELECT mb_per_active_day AS m,
    NTILE(4) OVER(ORDER BY mb_per_active_day) AS q
    FROM customer_features
    WHERE usage_days > 0)

Select q, COUNT(*) AS subs,
    ROUND(MIN(m), 2) AS min_mb,
    ROUND(MAX(m), 2) AS max_mb
FROM n
GROUP BY q
ORDER BY q

""")

,q,subs,min_mb,max_mb
0,1,5499,25.46,657.32
1,2,5499,657.35,1092.83
2,3,5499,1092.84,1571.87
3,4,5499,1572.12,9719.60


In [71]:
q("""
    WITH ranked AS (
        SELECT
            NTILE(4) OVER (ORDER BY mb_per_active_day) AS quartile,
            mb_per_active_day
        FROM customer_features
        WHERE mb_per_active_day > 0
    )
    SELECT
        quartile,
        COUNT(*) AS subscribers,
        MIN(mb_per_active_day) AS min_mb_per_active_day,
        MAX(mb_per_active_day) AS max_mb_per_active_day
    FROM ranked
    GROUP BY quartile
    ORDER BY quartile
""")

,quartile,subscribers,min_mb_per_active_day,max_mb_per_active_day
0,1,5499,25.46,657.32
1,2,5499,657.35,1092.83
2,3,5499,1092.84,1571.87
3,4,5499,1572.12,9719.60


In [72]:
#E4 · For each plan, how much of the monthly data allowance does a subscriber use?
#STEPS
#1. Group by plan_name, carrying the monthly_gb allowance from v_plans_clean.
#2. Convert each subscriber's six-month data total to gigabytes per month.
#3. Average it per plan.
#4. Sort by allowance so the two columns can be read against each other.
q("""
 SELECT plan_name, monthly_gb,
 ROUND(AVG(total_data_mb) / 1024, 2) AS avg_gb
 FROM customer_features
 WHERE usage_days > 0
 GROUP BY plan_name, monthly_gb
 ORDER BY monthly_gb;
""")

,plan_name,monthly_gb,avg_gb
0,Prepaid Lite 99,15.0,17.31
1,Prepaid Smart 155,30.0,34.31
2,Prepaid Value 209,30.0,35.33
3,Postpaid 399,40.0,47.13
4,Prepaid Max 479,45.0,53.06
5,Prepaid Popular 299,45.0,53.59
6,Prepaid Annual 1799,60.0,71.22
7,Prepaid Quarterly 719,60.0,70.67
8,Postpaid 499,75.0,92.82
9,Prepaid Data 349,75.0,90.76


In [73]:
#Does voice use per day differ by connection type and device, as data use does?

#STEPS
#1. Group by connection_type and device_type.
#2. Compute voice minutes per active day and megabytes per active day side by side.
#3. Sort by the voice figure.

q("""
select
    connection_type,
    device_type,
    count(*) as loading,
    round(avg(total_voice_min * 1.0 / nullif(usage_days, 0)), 2) as voice_per_day,
    round(avg(mb_per_active_day), 1) as mb_per_day
from customer_features
where usage_days > 0
group by connection_type, device_type
order by voice_per_day desc;
""")

,connection_type,device_type,loading,voice_per_day,mb_per_day
0,Postpaid,Feature Phone,690,48.61,530.7
1,Postpaid,Smartphone,4161,47.95,3352.8
2,Prepaid,Smartphone,14807,47.88,1093.1
3,Prepaid,Feature Phone,2338,47.86,166.4


In [74]:
#Do the subscribers who spend the most also use the most data?
#STEPS
#1. Group customer_features by m_score, the monetary quartile built in Step 2.
#2. Average spend, data per active day and churn together.
q("""
SELECT
    m_score,
    COUNT(*) AS subs,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu,
    ROUND(AVG(mb_per_active_day), 2) AS avg_mb_per_active_day,
    ROUND(AVG(is_churned), 2) AS avg_churn
FROM customer_features
GROUP BY m_score
ORDER BY m_score;
""")

,m_score,subs,avg_actual_arpu,avg_mb_per_active_day,avg_churn
0,1,5500,27.79,972.57,0.55
1,2,5500,157.54,778.61,0.20
2,3,5500,279.68,1116.54,0.11
3,4,5500,912.84,2749.78,0.09


In [75]:
#· How many SMS does a subscriber send per day, and how many never send one?
#S T E P S
#1. Average SMS per active day across everybody with usage.
#2. Count how many never sent one at all.

q("""
SELECT
    ROUND(AVG(100*total_sms/nullif(usage_days,0)), 2) AS avg_sms_per_active_day,
    SUM(total_sms = 0) AS num_never_sent_sms,
    count(*) AS total_subscribers
FROM customer_features
WHERE usage_days > 0;
""")

,avg_sms_per_active_day,num_never_sent_sms,total_subscribers
0,59.41,654,21996


In [76]:
#F1 · How common is roaming, counted in days and in subscribers?
#STEPS
#1. Read usage_daily, where one row is one subscriber-day.
#2. Group by is_roaming.
#3. Count days, express as a share, and count distinct subscribers.
q("""
SELECT
    is_roaming,
    COUNT(*) AS num_days,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM usage_daily), 2) AS perc_days,
    COUNT(DISTINCT customer_id) AS num_subscribers
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,num_days,perc_days,num_subscribers
0,0,1287143,97.02,21994
1,1,39596,2.98,16233


In [77]:
#F2 · Do subscribers use more or less on roaming days than at home?
#STEPS
#1. Same table, same grouping.
#2. Average data_mb and voice_minutes in each group.
#3. Read the two rows against each other.
q("""
SELECT
    is_roaming,
    ROUND(AVG(data_mb), 2) AS avg_data_mb,
    ROUND(AVG(voice_minutes), 2) AS avg_voice_minutes
FROM usage_daily
GROUP BY is_roaming;
""")

,is_roaming,avg_data_mb,avg_voice_minutes
0,0,1572.95,56.27
1,1,1570.05,56.09


In [78]:
#F3 · Do postpaid subscribers roam more than prepaid ones?
#STEPS
#1. Move to customer_features, where roaming_days is already counted per subscriber.
#2. Group by connection_type.
#3. Average the roaming days and count what share ever roamed at all.
q("""
SELECT
    connection_type,
    ROUND(AVG(roaming_days), 2) AS avg_roaming_days,
    COUNT(CASE WHEN roaming_days > 0 THEN 1 END) AS num_roamed,
    ROUND(COUNT(CASE WHEN roaming_days > 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS perc_roamed
FROM customer_features
GROUP BY connection_type;
""")

,connection_type,avg_roaming_days,num_roamed,perc_roamed
0,Postpaid,1.84,3630,74.83
1,Prepaid,1.79,12603,73.49


In [79]:
#F5 · Which five circles roam the most?
#STEPS
#1. Group customer_features by circle.
#2. Divide total roaming days by total usage days, so circles of different sizes are comparable.
#3. Sort and take five.
q("""
SELECT
    circle_name,
    ROUND(100*SUM(roaming_days) * 1.0 / SUM(usage_days), 2) AS roaming_per_usage_day,
    COUNT(*) AS subs
FROM customer_features
GROUP BY circle_name
ORDER BY roaming_per_usage_day DESC
LIMIT 5;
""")


,circle_name,roaming_per_usage_day,subs
0,Odisha,3.07,696
1,Haryana,3.07,1041
2,Delhi,3.07,735
3,Karnataka,3.05,1361
4,Assam,3.05,1301


In [80]:
#F4 · Do subscribers who roam more often leave less often?
#STEPS
#1. Band roaming_days into four groups with CASE WHEN.
#2. Count and average churn in each.
#3. Read the gradient — and then be suspicious of it.
q("""
SELECT
    CASE
        WHEN roaming_days = 0 THEN '0 days'
        WHEN roaming_days BETWEEN 1 AND 5 THEN '1-5 days'
        WHEN roaming_days BETWEEN 6 AND 15 THEN '6-15 days'
        ELSE '16+ days'
    END AS roaming_band,
    COUNT(*) AS subs,
    ROUND(AVG(is_churned), 2) AS avg_churn
FROM customer_features
GROUP BY roaming_band
""")

,roaming_band,subs,avg_churn
0,0 days,5767,0.45
1,1-5 days,15468,0.17
2,6-15 days,765,0.04


In [81]:
#G1 · What do subscribers complain about most?
#STEPS
#1. Read complaints, where one row is one ticket.
#2. Group by category.
#3. Count and share.
#4. Sort biggest first.
q("""
SELECT category,
    COUNT(*) AS tickets,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints), 2) AS share_pct
FROM complaints
GROUP BY category
ORDER BY tickets DESC
""")



,category,tickets,share_pct
0,Slow Data,8035,29.94
1,Dropped Call,6365,23.71
2,Network Coverage,4885,18.20
3,Billing,3738,13.93
4,Recharge Issue,2679,9.98
5,SIM/KYC,1139,4.24


In [82]:
#G2 · Does the complaint channel change how often a ticket is resolved?
#STEPS
#1. Group complaints by channel.
#2. Count.
#3. Compute the resolved share as an average of is_resolved times 100.
q("""
SELECT channel,
    COUNT(*) AS tickets,
    ROUND(100.0 * AVG(is_resolved), 2) AS resolved_pct
FROM complaints
GROUP BY channel
""")

,channel,tickets,resolved_pct
0,Airtel App,6616,70.04
1,Call Center,6728,69.72
2,Store,6799,70.33
3,Twitter/X,6698,70.36


In [83]:
# G3 · How many tickets are still open?
# STEPS
# 1. Group by is_resolved.
# 2. Count and share.
q("""
SELECT is_resolved,
    COUNT(*) AS tickets,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints), 2) AS share_pct
FROM complaints
GROUP BY is_resolved
""")

,is_resolved,tickets,share_pct
0,0,8021,29.88
1,1,18820,70.12


In [84]:
# Which complaint category is resolved least often, and most slowly?
# STEPS
# 1. Group by category.
# 2. Compute resolved share and average resolution time.
q("""
SELECT
    category,
    ROUND(100.0 * AVG(is_resolved), 2) AS resolved_pct,
    ROUND(AVG(resolution_days), 2) AS avg_resolution_tIME
FROM complaints
GROUP BY category
ORDER BY resolved_pct ASC, avg_resolution_Time DESC
""")

,category,resolved_pct,avg_resolution_tIME
0,SIM/KYC,68.57,7.65
1,Network Coverage,69.52,7.60
2,Dropped Call,69.71,7.46
3,Slow Data,69.97,7.55
4,Billing,70.87,7.50
5,Recharge Issue,72.23,7.55


In [85]:
# 2 G5 · How many days does a resolved ticket take to close?
# STEPS
# 1. Filter to resolved tickets only.
# 2. Average, minimum and maximum resolution_days.
# 3. Say explicitly what population that is measured on.
q("""
SELECT
    AVG(resolution_days) AS avg_days,
    MIN(resolution_days) AS min_days,
    MAX(resolution_days) AS max_days
FROM complaints
WHERE is_resolved = 1
""")

,avg_days,min_days,max_days
0,7.536929,1.0,14.0


In [86]:
# How many subscribers have raised 0, 1, 2 or more complaints?
# STEPS
# 1. Group customer_features by complaint_count.
# 2. Count subscribers at each level.
# 3. Read the whole distribution rather than the average.
q("""
SELECT
    complaint_count,
    COUNT(*) AS subscribers
FROM customer_features
GROUP BY complaint_count
ORDER BY complaint_count
""")

,complaint_count,subscribers
0,0,6631
1,1,7854
2,2,4726
3,3,1926
4,4,625
5,5,183
6,6,44
7,7,10
8,8,1


In [87]:
# · Which five circles raise the most complaints per subscriber?
# STEPS
# 1. Group by circle_name.
# 2. Divide total tickets by subscribers so circles of different sizes compare.
# 3. Sort and take five
q("""
SELECT circle_name,
    COUNT(*) AS subs,
    ROUND(1.0 * SUM(complaint_count) / COUNT(*), 2) AS perc_per_subs
    FROM customer_features
    GROUP BY circle_name
    ORDER BY perc_per_subs DESC
    LIMIT 5;""")

,circle_name,subs,perc_per_subs
0,Tamil Nadu,731,1.29
1,Bihar,887,1.29
2,Punjab,558,1.27
3,Kolkata,572,1.27
4,Delhi,735,1.27


In [88]:
# H1 · How many of our current subscribers joined in each year?
# STEPS
# 1. Read customer_features.
# 2. Pull the year out of activation_date with strftime('%Y', …).
# 3. Group and count.
q("""SELECT strftime('%Y', activation_date) AS year,
    COUNT(*) AS subs
    FROM customer_features
    GROUP BY year;""")

,year,subs
0,2022,6922
1,2023,6925
2,2024,7053
3,2025,1100


In [89]:
# H2 · What share of each joining year has left?
# STEPS
# 1. Same grouping.
# 2. Add the churn count and rate.
# 3. Read down the churn column.
q("""SELECT strftime('%Y', activation_date) AS year,
    COUNT(*) AS subs,
    SUM(is_churned) AS churn_count,
    ROUND(100.0 * SUM(is_churned) / COUNT(*), 2) AS churn_rate
    FROM customer_features
    GROUP BY year;""")

,year,subs,churn_count,churn_rate
0,2022,6922,1374,19.85
1,2023,6925,1632,23.57
2,2024,7053,1901,26.95
3,2025,1100,319,29.00


In [90]:
# H3 · How does churn change with each completed year on the network?
# STEPS
# 1. Group by tenure_years
# 2. Sort by tenure
q("""SELECT tenure_years,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn
    FROM customer_features
    GROUP BY tenure_years
    ORDER BY tenure_years;""")

,tenure_years,subs,churn
0,0,4610,28.81
1,1,6961,24.78
2,2,7055,21.84
3,3,3374,18.73


In [91]:
# H4 · How long had a leaver been with us when they left?
# STEPS
# 1. Filter to churned subscribers.
# 2. Average, minimum and maximum days_to_churn, which Step 2 built from the two dates

q("""SELECT COUNT(*) AS churned_subscribers,
    AVG(days_to_churn) AS avg_days,
    MIN(days_to_churn) AS min_days,
    MAX(days_to_churn) AS max_days
    FROM customer_features
    WHERE is_churned = 1;""")

,churned_subscribers,avg_days,min_days,max_days
0,5226,567.183123,10,1259


In [92]:
# H5 · Do longer-standing subscribers pay more?
# STEPS
# 1. Group by tenure_years.
# 2. Average both actual_arpu_6m and recharge_total_6m.
q("""SELECT tenure_years,
    COUNT(*) AS subs,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_arpu,
    ROUND(AVG(recharge_total_6m), 2) AS avg_recharge
    FROM customer_features
    GROUP BY tenure_years
    ORDER BY tenure_years;""")

,tenure_years,subs,avg_arpu,avg_recharge
0,0,4610,329.10,1974.63
1,1,6961,342.72,2056.30
2,2,7055,350.13,2100.76
3,3,3374,357.22,2143.29


In [93]:
# 1 · How many subscribers have we lost, and what is the churn rate?
# S T E P S
# 1. Count all subscribers.
# 2. Sum is_churned.
# 3. Average it for the rate.

q("""SELECT COUNT(*) AS total_subscribers,
    SUM(is_churned) AS churned_subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features;""")

,total_subscribers,churned_subscribers,churn_rate
0,22000,5226,23.75


In [94]:
# I2 · What is the churn rate for prepaid and for postpaid subscribers?
# STEPS
# 1. Group by connection_type.
# 2. Count, sum and average churn.

q("""SELECT connection_type,
    COUNT(*) AS subs,
    SUM(is_churned) AS churned_subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features
    GROUP BY connection_type;""")

,connection_type,subs,churned_subscribers,churn_rate
0,Postpaid,4851,785,16.18
1,Prepaid,17149,4441,25.90


In [95]:
# I3 · Which six plans have the highest churn rate?
# STEPS
# 1. Group by plan_name, carrying plan_type.
# 2. Count and average churn.
# 3. Sort worst first, take six.
q("""SELECT plan_name,plan_type,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn
    FROM customer_features
    GROUP BY plan_name, plan_type
    ORDER BY churn DESC
    LIMIT 6;""")

,plan_name,plan_type,subs,churn
0,Prepaid Lite 99,Prepaid,996,31.12
1,Prepaid Smart 155,Prepaid,1137,30.17
2,Prepaid Entertainment 399,Prepaid,2142,26.24
3,Prepaid Quarterly 719,Prepaid,2561,25.85
4,Prepaid Value 209,Prepaid,1437,25.26
5,Prepaid Annual 1799,Prepaid,2901,25.13


In [96]:
# I4 · Which circles have the highest and the lowest churn rate?
# STEPS
# 1. Group by circle_name, carrying region.
# 2. Count and average churn.
# 3. Sort worst first and print all 22, because the spread is the finding.
q("""SELECT circle_name,region,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features
    GROUP BY circle_name, region
    ORDER BY churn_rate DESC;""")

,circle_name,region,subs,churn_rate
0,Himachal Pradesh,North,1247,33.44
1,Maharashtra,West,1428,32.91
2,Andhra Pradesh,South,1412,28.90
3,Mumbai,West,1395,28.10
4,Uttar Pradesh (East),North,554,27.26
5,Haryana,North,1041,25.46
6,Jammu & Kashmir,North,933,24.87
7,Gujarat,West,1210,24.63
8,Assam,East,1301,23.14
9,Uttar Pradesh (West),North,1081,22.94


In [97]:
# I5 · Do smartphone users leave at a different rate from feature-phone users?
# STEPS
# 1. Group by device_type.
# 2. Count and average churn.
# 3. Read the two numbers.
q("""SELECT device_type,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features
    GROUP BY device_type;""")

,device_type,subs,churn_rate
0,Feature Phone,3029,23.74
1,Smartphone,18971,23.76


In [98]:
# I6 · Does churn differ by acquisition channel?
# STEPS
# 1. Group by acquisition_channel.
# 2. Count and average churn.
# 3. Sort worst first.
q("""SELECT acquisition_channel,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features
    GROUP BY acquisition_channel
    ORDER BY churn_rate DESC;""")

,acquisition_channel,subs,churn_rate
0,Call Center,1770,24.63
1,Airtel App,6513,24.20
2,Website,3081,24.12
3,Doorstep,1349,23.80
4,Retailer,9287,23.15


In [99]:
# I7 · Does churn differ by age band?
# STEPS
# 1. Group by age_band.
# 2. Count and average churn.
# 3. Sort by band.

q("""SELECT age_band,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features
    GROUP BY age_band
    ORDER BY age_band ASC;""")

,age_band,subs,churn_rate
0,18-24,5268,23.65
1,25-34,7467,23.70
2,35-44,4824,23.71
3,45-54,2895,23.49
4,55+,1546,24.97


In [100]:
# J1 · Does an open complaint make a subscriber more likely to leave?
# STEPS
# 1. Use CASE WHEN to build three groups from complaint_count and unresolved_count.
# 2. Count and average churn in each.
# 3. Sort so the three read in order.
q("""SELECT CASE WHEN complaint_count=0
            THEN 'never complained'
            WHEN unresolved_count=0
            THEN 'all resolved'
            ELSE 'open ticket'
       END AS grp,
       COUNT(*) AS subs,
       ROUND(100.0* AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY grp
ORDER BY grp ;""")

,grp,subs,churn
0,all resolved,8689,23.75
1,never complained,6631,20.03
2,open ticket,6680,27.46


In [101]:
# · Does churn rise with each extra open ticket?
# S T E P S
# 1. Group by unresolved_count directly.
# 2. Count and average churn.
# 3. Read the gradient and then read the group sizes.
q("""SELECT unresolved_count,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features
    GROUP BY unresolved_count
    ;""")

,unresolved_count,subs,churn_rate
0,0,15320,22.14
1,1,5515,26.07
2,2,1018,33.10
3,3,120,37.50
4,4,25,48.00
5,5,2,100.00


In [102]:
# K1 · What is the churn rate in each recency quartile?
# S T E P S
# 1. Group by r_score.
# 2. Average churn.
# 3. Look at it before saying anything.
q("""SELECT r_score,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features
    GROUP BY r_score
    ORDER BY r_score ASC;""")

,r_score,subs,churn_rate
0,1,5500,91.49
1,2,5500,3.53
2,3,5500,0.00
3,4,5500,0.00


In [103]:
# K2 · What is the churn rate in each frequency quartile?
# STEPS
# 1. Group by f_score.
# 2. Average recharge_count to check the score means what it says.
# 3. Average churn.
q("""SELECT f_score,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn
    FROM customer_features
    GROUP BY f_score
    ORDER BY f_score ASC;""")

,f_score,subs,churn
0,1,5500,45.44
1,2,5500,38.67
2,3,5500,7.76
3,4,5500,3.15


In [104]:
# K3 · What are the spend and the churn rate in each spend quartile?
# STEPS
# 1. Group by m_score.
# 2. Average spend and churn.
q("""SELECT m_score,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn
    FROM customer_features
    GROUP BY m_score
    ORDER BY m_score ASC;""")

,m_score,subs,churn
0,1,5500,54.82
1,2,5500,19.96
2,3,5500,11.24
3,4,5500,9.00


In [105]:
# L1 · What are the churn rate and the monthly revenue of each region?
# STEPS
# 1. Group by region.
# 2. Count, sum revenue and average churn.
# 3. Sort by churn.
q("""SELECT region,
    COUNT(*) AS subs,
    ROUND(SUM(actual_arpu_6m), 2) AS total_revenue,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate
    FROM customer_features
    GROUP BY region
    ORDER BY churn_rate DESC;""")

,region,subs,total_revenue,churn_rate
0,West,4665,1571780.02,27.85
1,North,7231,2477493.21,23.73
2,East,5459,1893199.66,21.91
3,South,4645,1635737.25,21.85


In [106]:
# L2 · Which four circles churn the most, and what do their subscribers pay?
# STEPS
# 1. Group by circle and region.
# 2. Average churn and ARPU together.
# 3. Sort worst first, take four.
q("""SELECT circle_name,region,
    COUNT(*) AS subs,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_arpu
    FROM customer_features
    GROUP BY circle_name, region
    ORDER BY churn_rate DESC
    LIMIT 4;""")

,circle_name,region,subs,churn_rate,avg_arpu
0,Himachal Pradesh,North,1247,33.44,313.16
1,Maharashtra,West,1428,32.91,315.15
2,Andhra Pradesh,South,1412,28.90,328.17
3,Mumbai,West,1395,28.10,339.27


In [107]:
# L7 · Do regions differ in revenue per subscriber and in data use?
# S T E P S
# 1. Group by region.
# 2. Average actual_arpu_6m and mb_per_active_day.
# 3. Sort by ARPU.
q("""SELECT region,
    COUNT(*) AS subs,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_arpu,
    ROUND(AVG(mb_per_active_day), 2) AS avg_data_per_day
    FROM customer_features
    GROUP BY region
    ORDER BY avg_arpu DESC;""")



,region,subs,avg_arpu,avg_data_per_day
0,South,4645,352.15,1430.50
1,East,5459,346.80,1404.47
2,North,7231,342.62,1388.55
3,West,4665,336.93,1403.09


In [108]:
# · How many subscribers are in group A and in group B?
# S T E P S
# 1. Group customer_features by ab_group.
# 2. Count and share.
q("""SELECT ab_group,
    COUNT(*) AS subs,
    ROUND(100.0 * COUNT(*) / 22000, 2) AS perc
    FROM customer_features
    GROUP BY ab_group;""")

,ab_group,subs,perc
0,A,10996,49.98
1,B,11004,50.02


In [109]:
# What is the churn rate in group A and in group B?
# STEPS
# 1. Group by ab_group.
# 2. Count, sum and average is_churned.
# 3. Read the two rates and then say nothing conclusive.
q("""SELECT ab_group,
    COUNT(*) AS subs,
    SUM(is_churned) AS churned,
    ROUND(100.0 * SUM(is_churned) / COUNT(*), 2) AS churn_rate
    FROM customer_features
    GROUP BY ab_group;""")

,ab_group,subs,churned,churn_rate
0,A,10996,2587,23.53
1,B,11004,2639,23.98


In [110]:
# Does revenue per subscriber differ between the groups, on either ARPU column?
# 1. Group by ab_group.
# 2. Average both the stated and the measured ARPU column.
# 3. Compare the two rows, and then compare the two columns.
q("""SELECT ab_group,
    COUNT(*) AS subs,
    ROUND(AVG(stated_arpu), 2) AS avg_stated_arpu,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu
    FROM customer_features
    GROUP BY ab_group;""")

,ab_group,subs,avg_stated_arpu,avg_actual_arpu
0,A,10996,424.30,347.58
1,B,11004,418.99,341.35
